In [ ]:
#The purpose of this notebook is to replicate the results of Ghorbani et al. results for Dmax regression using XGBoost.
#Ghorbani used a RandomForestRegressor(max_depth=12, n_estimators=45) from scikit-learn

In [29]:
#import the required libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import re
from sklearn.model_selection import RepeatedKFold,ShuffleSplit
from CBFV import composition
from scipy.stats import sem

In [2]:
#Take formula column and parse it into a dataframe of element columns with atomic percentages as values. This function should be able to handle the complex formulas in the Ghorbani dataset, including nested parentheses, brackets, and braces, as well as fractions and equal splits.
def assemble_composition_df(df, formula_column):
    element_list = [
        "Ag", "Al", "Am", "As", "Au",
        "B", "Ba", "Be", "Bi",
        "C", "Ca", "Cd", "Ce", "Co", "Cr", "Cs", "Cu",
        "Dy",
        "Er", "Eu",
        "Fe",
        "Ga", "Gd", "Ge",
        "H", "Hf", "Hg", "Ho",
        "In", "Ir",
        "K",
        "La", "Li", "Lu",
        "Mg", "Mn", "Mo",
        "N", "Na", "Nb", "Nd", "Ni", "Np",
        "O", "Os",
        "P", "Pa", "Pb", "Pd", "Pr", "Pt", "Pu",
        "Rb", "Re", "Rh", "Ru",
        "S", "Sb", "Sc", "Se", "Si", "Sm", "Sn", "Sr",
        "Ta", "Tb", "Tc", "Te", "Th", "Ti", "Tl", "Tm",
        "U",
        "V",
        "W",
        "Y", "Yb",
        "Zn", "Zr"
    ]
    
    def parse_fraction(s):
        """Parse a string that might be a fraction (e.g., '5/6') or a number."""
        if '/' in s:
            num, denom = s.split('/')
            return float(num) / float(denom)
        return float(s)
    
    def parse_element_composition(formula_str):
        """
        Parse element-number pairs from a formula string.
        Returns a dict of {element: amount}
        """
        composition = {}
        # Pattern to match element followed by optional number (including fractions)
        pattern = r'([A-Z][a-z]?)(\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?)?'
        
        matches = re.findall(pattern, formula_str)
        for element, amount in matches:
            if element and element in element_list:
                if amount:
                    val = parse_fraction(amount)
                else:
                    val = 1.0
                composition[element] = composition.get(element, 0) + val
        
        return composition
    
    def parse_formula(formula):
        """
        Parse a complete alloy formula handling nested brackets, parentheses, and braces.
        Returns a dict of {element: atomic_percent}
        """
        composition = {}
        
        # Remove citation references like [24], [30], etc. at the end
        formula = re.sub(r'\[\d+\]$', '', formula)
        formula = re.sub(r'\[\d+\]', '', formula)
        
        # Remove spaces and commas used as separators
        formula = formula.replace(' ', '').replace(',', '')
        
        def process_innermost_group(f):
            """Find and process the innermost bracketed group."""
            pattern = r'([\(\[\{])([^\(\)\[\]\{\}]+)([\)\]\}])(\d+(?:\.\d+)?)?'
            
            match = re.search(pattern, f)
            if not match:
                return f, False
            
            open_bracket, content, close_bracket, multiplier = match.groups()
            
            # Parse the content of the group
            inner_comp = parse_element_composition(content)
            
            # Calculate the sum of inner compositions
            inner_sum = sum(inner_comp.values())
            
            # Determine the multiplier
            if multiplier:
                mult = float(multiplier)
            else:
                mult = 1.0
            
            # Determine if inner values are fractions or percentages
            # If sum is close to 1, treat as fractions; if close to 100, treat as percentages
            if inner_sum > 1.5:  # Likely percentages within the group
                # Normalize to fractions, then multiply
                inner_comp = {k: v / inner_sum for k, v in inner_comp.items()}
            
            # Apply multiplier
            expanded = {elem: amt * mult for elem, amt in inner_comp.items()}
            
            # Create replacement string
            replacement_parts = []
            for elem, amt in expanded.items():
                replacement_parts.append(f"{elem}{amt}")
            replacement = ''.join(replacement_parts)
            
            new_f = f[:match.start()] + replacement + f[match.end():]
            
            return new_f, True
        
        # Iteratively process innermost groups until none remain
        processed_formula = formula
        max_iterations = 20
        iteration = 0
        
        while iteration < max_iterations:
            processed_formula, found = process_innermost_group(processed_formula)
            if not found:
                break
            iteration += 1
        
        # Now parse the final expanded formula
        composition = parse_element_composition(processed_formula)
        
        # Handle equal split case (elements with no numbers)
        total = sum(composition.values())
        num_elements = len(composition)
        
        # Check if all elements have value 1.0 (no numbers given)
        if num_elements > 0 and all(v == 1.0 for v in composition.values()):
            equal_share = 100.0 / num_elements
            composition = {k: equal_share for k in composition}
        # If total is very small (< 2), scale up to 100
        elif total > 0 and total < 2:
            scale = 100.0 / total
            composition = {k: v * scale for k, v in composition.items()}
        
        return composition
    
    # Process all formulas
    composition_dicts = []
    for formula in df[formula_column]:
        try:
            comp = parse_formula(str(formula))
            composition_dicts.append(comp)
        except Exception as e:
            print(f"Error parsing '{formula}': {e}")
            composition_dicts.append({})
    
    # Create DataFrame with element columns
    comp_df = pd.DataFrame(composition_dicts)
    
    # Ensure all element columns exist, fill missing with 0
    for elem in element_list:
        if elem not in comp_df.columns:
            comp_df[elem] = 0.0
    
    # Reorder columns to match element_list and fill NaN with 0
    comp_df = comp_df.reindex(columns=element_list, fill_value=0.0)
    comp_df = comp_df.fillna(0.0)
    
    return comp_df

#take composition df and produce composition strings
def canonical_comp_string(df, tol=1e-9, decimals=2):
    element_cols = sorted([c for c in df.columns if c != "Composition String"])
    out = []
    for _, row in df[element_cols].iterrows():
        vals = row.astype(float).fillna(0.0).to_numpy()
        vals[vals < tol] = 0.0
        s = vals.sum()
        if s <= 0:
            out.append("")
            continue
        vals = vals / s * 100.0
        vals = np.round(vals, decimals)
        parts = [f"{el}{v:.{decimals}f}" for el, v in zip(element_cols, vals) if v > 0]
        out.append("".join(parts))
    return out


In [3]:
#load the Ghorbani dataset
raw_data = pd.read_excel(r"Data\Paper Data\Ghorbani, 2022.xlsx")
raw_data.columns

Index(['No.', 'Alloy', 'Tg', 'Tx', 'Tl', 'X1 (Trg)', 'X2 (Delta T)',
       'X3 (Alpha)', 'X4 (Beta)', 'X5 (new Beta)', 'X6 (Gamma)',
       'X7 (Gamma m)', 'X8 (Delta)', 'X9 (NULL sign)', 'X10 (Omega)',
       'X11 (Omega m)', 'X12 (Theta)', 'X13 (Xi)', 'X14 (Beta Prime)',
       'X15 (Dleta Trg)', 'X16 (Gp)', 'X17 (Gamma C)', 'Y (Dmax)'],
      dtype='object')

In [4]:
#create composition dataframe from the raw data
composition_df = assemble_composition_df(raw_data, "Alloy")

#find entries that have totals greater than 100
over_100 = composition_df.sum(axis=1) > 100
over_100_count = sum(over_100)
print(f"Number of entries with totals greater than 100: {over_100_count}")

#drop number of entries with totals greater than 100
n_before = len(composition_df)
composition_df = composition_df[~over_100]
n_after = len(composition_df)
print(f"Entries before dropping: {n_before}, Entries after dropping: {n_after}")

#use the same over 100 index to drop from the raw data
raw_data = raw_data[~over_100]

Number of entries with totals greater than 100: 26
Entries before dropping: 715, Entries after dropping: 689


In [5]:
#assemble composition strings and add to the raw data dataframe
canonical_comp_strings = canonical_comp_string(composition_df)
raw_data["Composition String"] = canonical_comp_strings


#check raw data frame for duplicate composition strings
duplicates = raw_data["Composition String"].duplicated().sum()
print(f"Number of duplicate composition strings: {duplicates}")
len_pre_dup = len(raw_data)

#replace duplicates with average values
raw_data = raw_data.groupby("Composition String").mean(numeric_only=True).reset_index()
len_post_dup = len(raw_data)
print(f"Entries before removing duplicates: {len_pre_dup}, Entries after removing duplicates: {len_post_dup}")

Number of duplicate composition strings: 28
Entries before removing duplicates: 689, Entries after removing duplicates: 661


In [6]:
raw_data.columns

Index(['Composition String', 'No.', 'Tg', 'Tx', 'Tl', 'X1 (Trg)',
       'X2 (Delta T)', 'X3 (Alpha)', 'X4 (Beta)', 'X5 (new Beta)',
       'X6 (Gamma)', 'X7 (Gamma m)', 'X8 (Delta)', 'X9 (NULL sign)',
       'X10 (Omega)', 'X11 (Omega m)', 'X12 (Theta)', 'X13 (Xi)',
       'X14 (Beta Prime)', 'X15 (Dleta Trg)', 'X16 (Gp)', 'X17 (Gamma C)',
       'Y (Dmax)'],
      dtype='object')

In [7]:
#split the raw data into features and target
X = raw_data.copy().drop(['No.', 'Composition String','Y (Dmax)'], axis=1)
y = raw_data['Y (Dmax)']

#separate into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#replicate the random forest regression model and the cv used by Ghorbani et al. which is a repeated k-fold with 5 splits and 20 repeats
cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=123)

#initiate the cross fold validation and store the R^2 scores and mean absolute error scores for each fold
r2_scores = []
mae_scores = []

for fold_train, fold_test in cv.split(X_train):
    #pull the fold train and test sets
    x_fold_train, x_fold_test = X_train.iloc[fold_train], X_train.iloc[fold_test]
    y_fold_train, y_fold_test = y_train.iloc[fold_train], y_train.iloc[fold_test]
    
    #initiate the regressor with the same parameters as Ghorbani et al.
    RFregress = RandomForestRegressor(max_depth= 21,n_estimators=36, random_state=False)
    
    #fit the regressor to the fold train set
    RFregress.fit(x_fold_train, y_fold_train)
    
    #evaluate the regressor on the fold test set
    y_pred = RFregress.predict(x_fold_test)
    
    #Calculate the R^2 score for the fold test set
    Fold_r2_score = RFregress.score(x_fold_test, y_fold_test)
    #calculate the mean absolute error for the fold test set
    Fold_mae = np.mean(np.abs(y_pred - y_fold_test))
    
    #store the scores for each fold
    r2_scores.append(Fold_r2_score)
    mae_scores.append(Fold_mae)
    
    
#print the average R^2 score and mean absolute error across all folds
avg_r2_score = np.mean(r2_scores)
avg_mae_score = np.mean(mae_scores)
print(f"Average R^2 score across all folds: {avg_r2_score:.4f}")
print(f"Average Mean Absolute Error across all folds: {avg_mae_score:.4f}")
    


Average R^2 score across all folds: 0.3169
Average Mean Absolute Error across all folds: 3.1283


In [27]:
#train the model on the full training set and evaluate on the test set
RFregress_Ghorbani = RandomForestRegressor(max_depth= 21,n_estimators=36, random_state=False)
RFregress_Ghorbani.fit(X_train, y_train)
y_pred = RFregress_Ghorbani.predict(X_test) 

#Calculate the R^2 score for the test set
Ghorbani_test_r2_score = RFregress_Ghorbani.score(X_test, y_test)
Ghorbani_test_mae_score = np.mean(np.abs(y_pred - y_test))
print(f"R^2 score on the test set: {Ghorbani_test_r2_score:.4f}")
print(f"Mean Absolute Error on the test set: {Ghorbani_test_mae_score:.4f}")

R^2 score on the test set: 0.1364
Mean Absolute Error on the test set: 3.5673


In [19]:
#apply the same model evaluation to calphad and CBFV data

#load the raw calphad data
raw_CAPHAD = pd.read_csv(r"Data\F(Composition)_Data\calphad_alloys_flattened_all_temps.csv")




In [20]:

#pull the composition strings from the processed ghorbani data and use them to filter the calphad data to only include entries that are in the ghorbani dataset
comp_strings = raw_data["Composition String"].tolist()
raw_CAPHAD = raw_CAPHAD[raw_CAPHAD["alloy_string"].isin(comp_strings)]

#filter the CALPHAD data to include only driving forces at the temepratures of interests
# Define the temperature ranges of interest
temperature_range = [2200, 1700, 2450, 2100, 2400, 1900, 1850]
temperature_range_str = [str(temp) for temp in temperature_range]

# Filter the columns to include only those with the specified temperature ranges and Df
filtered_CALPHAD_cols = [col for col in raw_CAPHAD.columns if any(temp in col for temp in temperature_range_str) and 'DF' in col]

print(f"Number of filtered CALPHAD columns: {len(filtered_CALPHAD_cols)}")

filtered_CALPHAD = raw_CAPHAD[filtered_CALPHAD_cols]
filtered_CALPHAD.head()

Number of filtered CALPHAD columns: 5628


,DF_AG2CA_T1700C,DF_AG2CA_T1850C,DF_AG2CA_T1900C,DF_AG2CA_T2100C,DF_AG2CA_T2200C,DF_AG2CA_T2400C,DF_AG2CA_T2450C,DF_AG3BE8_T1700C,DF_AG3BE8_T1850C,DF_AG3BE8_T1900C,...,DF_ZRSI2_T2200C,DF_ZRSI2_T2400C,DF_ZRSI2_T2450C,DF_ZRSI_T1700C,DF_ZRSI_T1850C,DF_ZRSI_T1900C,DF_ZRSI_T2100C,DF_ZRSI_T2200C,DF_ZRSI_T2400C,DF_ZRSI_T2450C
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
#Use the previously pulled compositions strings to produce a dataframe of composition strings for the CBFV data
comp_strings_df =  pd.DataFrame({'formula': comp_strings})
#add target colum for CBFV api
comp_strings_df['target'] = 0

#use the CBFV api to produce features for the composition strings
train_opt_CBFV_df, _, train_opt_formulae, skipped_train = composition.generate_features(comp_strings_df, elem_prop='magpie')

#print the number of skipped compositions for the CBFV data
print(f"Number of skipped compositions for CBFV: {len(skipped_train)}")

Processing Input Data: 100%|██████████| 661/661 [00:00<00:00, 34774.54it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 661/661 [00:00<00:00, 24470.07it/s]

	Creating Pandas Objects...
Number of skipped compositions for CBFV: 0


In [24]:
#combine the filtered CALPHAD data and the CBFV data into a single dataframe for model evaluation
combined_CBFV_CALPHAD_df = pd.concat([filtered_CALPHAD.reset_index(drop=True), train_opt_CBFV_df.reset_index(drop=True)], axis=1)


#split this combined dataframe into the same test and train sets as the original Ghorbani methodology using the index of the previoulsy split X_train and X_test dataframes
combined_X_train = combined_CBFV_CALPHAD_df.iloc[X_train.index]
combined_X_test = combined_CBFV_CALPHAD_df.iloc[X_test.index]

#the same y data can be used for the target variable since we are only changing the features used for the model evaluation

In [25]:
#test with the exact same model and cv as before but with the combined CBFV and CALPHAD features instead of the original features used by Ghorbani et al.
#replicate the random forest regression model and the cv used by Ghorbani et al. which is a repeated k-fold with 5 splits and 20 repeats
cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=123)

#initiate the cross fold validation and store the R^2 scores and mean absolute error scores for each fold
r2_CBFV_CALPHAD_scores = []
mae_CBFV_CALPHAD_scores = []

for fold_train, fold_test in cv.split(combined_X_train):
    #pull the fold train and test sets
    x_fold_train, x_fold_test = combined_X_train.iloc[fold_train], combined_X_train.iloc[fold_test]
    y_fold_train, y_fold_test = y_train.iloc[fold_train], y_train.iloc[fold_test]
    
    #initiate the regressor with the same parameters as Ghorbani et al.
    RFregress = RandomForestRegressor(max_depth= 21,n_estimators=36, random_state=False)
    
    #fit the regressor to the fold train set
    RFregress.fit(x_fold_train, y_fold_train)
    
    #evaluate the regressor on the fold test set
    y_pred = RFregress.predict(x_fold_test)
    
    #Calculate the R^2 score for the fold test set
    Fold_r2_score = RFregress.score(x_fold_test, y_fold_test)
    #calculate the mean absolute error for the fold test set
    Fold_mae = np.mean(np.abs(y_pred - y_fold_test))
    
    #store the scores for each fold
    r2_CBFV_CALPHAD_scores.append(Fold_r2_score)
    mae_CBFV_CALPHAD_scores.append(Fold_mae)
    
    
#print the average R^2 score and mean absolute error across all folds
avg_r2_score = np.mean(r2_CBFV_CALPHAD_scores)
avg_mae_score = np.mean(mae_CBFV_CALPHAD_scores)
print(f"Average R^2 score across all folds: {avg_r2_score:.4f}")
print(f"Average Mean Absolute Error across all folds: {avg_mae_score:.4f}")
    


Average R^2 score across all folds: 0.2913
Average Mean Absolute Error across all folds: 3.0841


In [28]:
#Evaluate on the test set with the combined features

RFregress_COMBINED = RandomForestRegressor(max_depth= 21,n_estimators=36, random_state=False)

RFregress_COMBINED.fit(combined_X_train, y_train)

#evaluate the regressor on the test set
y_pred = RFregress_COMBINED.predict(combined_X_test)

#Calculate the R^2 score for the test set
test_COMBINED_r2_score = RFregress_COMBINED.score(combined_X_test, y_test)
test_COMBINED_mae_score = np.mean(np.abs(y_pred - y_test))
print(f"R^2 score on the test set: {test_COMBINED_r2_score:.4f}")
print(f"Mean Absolute Error on the test set: {test_COMBINED_mae_score:.4f}")

R^2 score on the test set: 0.1862
Mean Absolute Error on the test set: 3.4146


In [ ]:
#optimize the model hyperparamters but using ax instead of the randomized search cv used by Ghorbani et al.
#def the evaluation funciton
def evaluate_parameters_RFR(paramters):
    
    #break the parameters out of the dictionary
    n_estimators = paramters['n_estimators']
    max_depth = paramters['max_depth']
    min_samples_split = paramters['min_samples_split']
    max_features = paramters['max_features']
    bootstrap = paramters['bootstrap']
    
    cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=123)

    #initiate the cross fold validation and store the R^2 scores and mean absolute error scores for each fold
    r2_fold_scores = []
    mae_fold_scores = []

    for fold_train, fold_test in cv.split(combined_X_train):
        #pull the fold train and test sets
        x_fold_train, x_fold_test = combined_X_train.iloc[fold_train], combined_X_train.iloc[fold_test]
        y_fold_train, y_fold_test = y_train.iloc[fold_train], y_train.iloc[fold_test]
        
        #initiate the regressor with the same parameters as Ghorbani et al.
        RFregress = RandomForestRegressor(max_depth= 21,n_estimators=36, random_state=False)
        
        #fit the regressor to the fold train se
        RFregress.fit(x_fold_train, y_fold_train)
        
        #evaluate the regressor on the fold test set
        y_pred = RFregress.predict(x_fold_test)
        
        #Calculate the R^2 score for the fold test set
        Fold_r2_score = RFregress.score(x_fold_test, y_fold_test)
        #calculate the mean absolute error for the fold test set
        Fold_mae = np.mean(np.abs(y_pred - y_fold_test))
        
        #store the scores for each fold
        r2_fold_scores.append(Fold_r2_score)
        mae_fold_scores.append(Fold_mae)
        
        
    #print the average R^2 score and mean absolute error across all folds
    avg_r2_score = np.mean(r2_fold_scores)
    avg_mae_score = np.mean(mae_fold_scores)
    print(f"Average R^2 score across all folds: {avg_r2_score:.4f}")
    print(f"Average Mean Absolute Error across all folds: {avg_mae_score:.4f}")